# Liu2024 Cohort Selection Sensitivity
**CLOSED SENSITIVITY.** Full50/Lv14/QC46 and selector/oracle analyses are complete. Do not rerun or tune cohort selection; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
from __future__ import annotations
raise RuntimeError('CLOSED cohort-selection sensitivity: see AGENTS.md section 2e.')
import builtins, hashlib, json, os, platform, random, sys
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import signal
from scipy.io import loadmat
from scipy.stats import kruskal, spearmanr
from sklearn.covariance import OAS
from sklearn.linear_model import Ridge
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | CWD: {Path.cwd()}")

# 2. Configuration
## 2.1 Prespecified Cohorts, Clinical Bins, and QC Defaults
See `notes/COHORT_SELECTION_PROTOCOL.md`; all bins, policies, selector features, and coverage levels are fixed before outcomes.
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-cohort-selection-sensitivity"),
    "analysis_artifact": "artifacts/liu2024-compact-mi-models/20260712_165746_790145_fd8ab986",
    "analysis_role": "locked_primary",
    "experiment_name": "cohort_selection_locked_shallow_primary",
    "config_note": "Locked Shallow OOF cohort-selection sensitivity; outcome-ranked oracle is invalid.",
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "participants_tsv": str(WORKING_DIR.parent / "Liu2024" / "participants.tsv"),
    "enabled": True,
    # ------------------------------------------------------------------
    # Dataset / fixed cohorts
    # ------------------------------------------------------------------
    "expected_subjects": [f"sub-{i:02d}" for i in range(1, 51)],
    "expected_trials_per_subject": 40,
    "lv14_subjects": ["sub-01","sub-03","sub-07","sub-09","sub-10","sub-11","sub-14","sub-15","sub-17","sub-29","sub-31","sub-32","sub-37","sub-41"],
    "native_sfreq": 500, "marker_channel_index": 32, "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300], "window_seconds": 4.0,
    "liu29_indices": list(range(17)) + list(range(18, 30)),
    "liu29_names": ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"],
    "motor13_names": ["F3","F4","FCz","FC3","FC4","Cz","C3","C4","CP3","CP4","Pz","P3","P4"],
    # ------------------------------------------------------------------
    # Clinical categories fixed before outcomes
    # ------------------------------------------------------------------
    "age_edges": [-1, 44, 64, 200], "age_labels": ["<45","45-64",">=65"],
    "duration_edges": [-1, 3, 7, 14, 100000], "duration_labels": ["1-3","4-7","8-14",">=15"],
    "nihss_edges": [-1, 0, 4, 15, 20, 42], "nihss_labels": ["0_none","1-4_minor","5-15_moderate","16-20_moderate_severe","21-42_severe"],
    "mbi_edges": [-1, 20, 60, 90, 99, 100], "mbi_labels": ["0-20_total","21-60_severe","61-90_moderate","91-99_slight","100_independent"],
    "mrs_labels": ["0","1","2","3","4","5"],
    # ------------------------------------------------------------------
    # Label-blind full-recording transductive QC
    # ------------------------------------------------------------------
    "qc_version": "liu2024_label_blind_qc_v1", "qc_bandpass_hz": [8.0, 30.0],
    "qc_welch_nperseg": 1000, "flat_scale_min": 0.05, "robust_scale_range": [0.1, 1000.0],
    "peak_to_peak_max": 5000.0, "extreme_robust_z": 20.0, "extreme_fraction_max": 0.01,
    "adc_centered_rail": 32760.0, "clipped_fraction_max": 0.001, "marker_valid_fraction_min": 0.95,
    "marker_fallback_max": 2, "cov_condition_max": 1e8, "cov_effective_rank_min": 2.0,
    "minimum_policy_coverage": 0.70,
    # ------------------------------------------------------------------
    # Nested patient selector / descriptive analyses
    # ------------------------------------------------------------------
    "selector_features": ["marker_fallback_count","robust_sd_median","peak_to_peak_median","extreme_fraction","flat_channel_count","line_50hz_ratio","hf_30_40_to_8_30_ratio","split_half_log_bandpower_reliability"],
    "selector_outer_folds": 5, "selector_inner_folds": 5, "ridge_alphas": [0.01,0.1,1.0,10.0,100.0],
    "coverage_levels": [1.0,0.9,0.8,0.7,0.6,0.5],
    "oracle_coverage_levels": [1.0,0.9,0.8,0.7,0.6,0.5,0.4,0.3,0.2,0.1],
    "trial_abstention_levels": [1.0,0.9,0.8,0.7,0.6,0.5,0.4,0.3,0.2,0.1],
    "oracle_demonstration_target_ba": 0.75,
    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "bootstrap_iterations": 10000, "seed": 20260713, "set_seed": True
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
if not CONFIG.get("enabled", True):
    raise RuntimeError("This optional sweep entry is disabled; set enabled=true explicitly to run it.")
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(encoding, errors="replace").decode(encoding, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep=kwargs.pop("sep"," " ); end=kwargs.pop("end","\n"); flush=kwargs.pop("flush",False); file=kwargs.pop("file",None)
    message=sep.join(str(arg) for arg in args); target=sys.stdout if file is None else file
    stamped=f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    _safe_write_text(target, stamped+end); _safe_write_text(_LOG_FILE_HANDLE, stamped+end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / "config.json"
config_path.write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device(); print(f"Using device: {DEVICE} (analysis is CPU-oriented)")
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True, warn_only=True)
BASE_SEED=int(CONFIG["seed"])
if CONFIG["set_seed"]: seed_everything(BASE_SEED); print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Locked OOF Artifact Loading and Exactly-Once Validation
## 3.2 Clinical Table and Prespecified Categories
## 3.3 Label-Blind Raw Signal QC and Signature Cache
QC summaries use each patient's full recording and are transductive. They are not strict fold-local QC.
## 3.4 Joined Analysis Table

In [ ]:
def resolve_repo_path(value):
    path=Path(value); return path if path.is_absolute() else WORKING_DIR/path
def bootstrap_ci(values, seed_offset=0):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]
    if not len(x): return [None,None]
    rng=np.random.default_rng(BASE_SEED+seed_offset); idx=rng.integers(0,len(x),size=(int(CONFIG["bootstrap_iterations"]),len(x)))
    return [float(v) for v in np.quantile(x[idx].mean(1),[0.025,0.975])]
def holm_adjust(p_values):
    p=np.asarray(p_values,float); out=np.full(len(p),np.nan); valid=np.flatnonzero(np.isfinite(p)); order=valid[np.argsort(p[valid])]; running=0.0
    for rank,idx in enumerate(order): running=max(running,(len(order)-rank)*p[idx]); out[idx]=min(1.0,running)
    return out
def json_safe(value):
    if isinstance(value,dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple,np.ndarray)): return [json_safe(v) for v in value]
    if isinstance(value,(np.integer,)): return int(value)
    if isinstance(value,(np.floating,)): return None if not np.isfinite(value) else float(value)
    if isinstance(value,float) and not np.isfinite(value): return None
    return value
def clinical_composition(frame):
    cols=["ParalysisSide","nihss_category","age_category","duration_category","mRS_category","mbi_category"]
    return {c:{str(k):int(v) for k,v in frame[c].astype(str).value_counts(dropna=False).sort_index().items()} for c in cols}

In [ ]:
print("Full CONFIG banner:\n"+json.dumps(CONFIG,indent=2,default=str))
SOURCE_ARTIFACT=resolve_repo_path(CONFIG["analysis_artifact"])
source_config=json.loads((SOURCE_ARTIFACT/"config.json").read_text(encoding="utf-8"))
if source_config.get("model_name","ShallowFBCSPNet") != "ShallowFBCSPNet": raise ValueError("Analysis artifact is not ShallowFBCSPNet")
folds=json.loads((SOURCE_ARTIFACT/"cv_results.json").read_text(encoding="utf-8"))
expected=CONFIG["expected_subjects"]; found=sorted({r["subject_id"] for r in folds})
if found != expected: raise AssertionError(f"Subject IDs mismatch: {found}")
trial_rows=[]; subject_rows=[]
for sid in expected:
    rows=sorted([r for r in folds if r["subject_id"]==sid],key=lambda r:r["fold_id"]); by_index={}
    for row in rows:
        n=len(row["test_indices"]); probs=np.asarray(row["probabilities"],float)
        if probs.shape != (n,2) or not np.isfinite(probs).all() or not np.allclose(probs.sum(1),1,atol=1e-5): raise AssertionError(f"Invalid probabilities for {sid}")
        for idx,y,pred,prob in zip(row["test_indices"],row["true_labels"],row["predictions"],probs):
            if int(idx) in by_index: raise AssertionError(f"Duplicate OOF trial {sid}/{idx}")
            by_index[int(idx)]=(int(y),int(pred),prob)
    if sorted(by_index) != list(range(CONFIG["expected_trials_per_subject"])): raise AssertionError(f"Not exactly-once 40 trials for {sid}")
    y=np.array([by_index[i][0] for i in range(40)]); pred=np.array([by_index[i][1] for i in range(40)])
    if set(y)!={0,1} or np.bincount(y,minlength=2).tolist()!=[20,20]: raise AssertionError(f"Labels not balanced for {sid}")
    subject_rows.append({"subject_id":sid,"balanced_accuracy":float(balanced_accuracy_score(y,pred)),"accuracy":float(np.mean(y==pred)),"n_trials":40})
    for i in range(40):
        prob=by_index[i][2]; trial_rows.append({"subject_id":sid,"trial_index":i,"true_label":by_index[i][0],"prediction":by_index[i][1],"probability_0":float(prob[0]),"probability_1":float(prob[1]),"confidence":float(prob.max())})
SUBJECT_BA=pd.DataFrame(subject_rows); TRIAL_OOF=pd.DataFrame(trial_rows)
published=pd.read_csv(SOURCE_ARTIFACT/"subject_results.csv").sort_values("subject_id")
if not np.allclose(SUBJECT_BA.sort_values("subject_id")["balanced_accuracy"],published["balanced_accuracy"]): raise AssertionError("Reconstructed BA differs from source artifact")
print(f"Validated {len(SUBJECT_BA)} subjects and {len(TRIAL_OOF)} exactly-once OOF trial predictions")

In [ ]:
clinical=pd.read_csv(CONFIG["participants_tsv"],sep="\t").rename(columns={"Participant_ID":"subject_id"})
if sorted(clinical["subject_id"].tolist()) != expected or clinical["subject_id"].duplicated().any(): raise AssertionError("participants.tsv must match Full50 one-to-one")
clinical["ParalysisSide"]=clinical["ParalysisSide"].fillna("Missing").astype(str)
clinical["age_category"]=pd.cut(clinical["Age"],CONFIG["age_edges"],labels=CONFIG["age_labels"],include_lowest=True)
clinical["duration_category"]=pd.cut(clinical["Duration"],CONFIG["duration_edges"],labels=CONFIG["duration_labels"],include_lowest=True)
clinical["nihss_category"]=pd.cut(clinical["NIHSS"],CONFIG["nihss_edges"],labels=CONFIG["nihss_labels"],include_lowest=True)
clinical["mbi_category"]=pd.cut(clinical["MBI"],CONFIG["mbi_edges"],labels=CONFIG["mbi_labels"],include_lowest=True)
clinical["mRS_category"]=pd.Categorical(clinical["mRS"].astype(str),categories=CONFIG["mrs_labels"],ordered=True)

In [ ]:
def raw_subject_path(sid): return Path(CONFIG["source_extract_dir"])/sid/f"{sid}_task-motor-imagery_eeg.mat"
def qc_signature():
    files=[raw_subject_path(s) for s in expected]
    payload={"version":CONFIG["qc_version"],"files":[(str(p.resolve()),p.stat().st_size) for p in files],"sfreq":CONFIG["native_sfreq"],"marker":[CONFIG["marker_channel_index"],CONFIG["onset_marker_value"],CONFIG["onset_plausible_range"]],"channels":CONFIG["liu29_names"],"bandpass":CONFIG["qc_bandpass_hz"],"thresholds":{k:CONFIG[k] for k in ["flat_scale_min","robust_scale_range","peak_to_peak_max","extreme_robust_z","extreme_fraction_max","adc_centered_rail","clipped_fraction_max","cov_condition_max","cov_effective_rank_min"]}}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()[:16],payload
def ratio(psd,freqs,num,den):
    a=psd[...,((freqs>=num[0])&(freqs<num[1]))].sum(-1); b=psd[...,((freqs>=den[0])&(freqs<den[1]))].sum(-1)
    return float(np.median(a/np.maximum(b,1e-20)))
def compute_subject_qc(sid):
    eeg=loadmat(raw_subject_path(sid))["eeg"][0,0]; raw=np.asarray(eeg["rawdata"],dtype=np.float64); x=raw[:,CONFIG["liu29_indices"],:]
    marker=raw[:,CONFIG["marker_channel_index"],:]; lo,hi=CONFIG["onset_plausible_range"]; candidates=[]
    for row in marker:
        hits=np.flatnonzero(row==CONFIG["onset_marker_value"]); hits=hits[(hits>=lo)&(hits<=hi)]; candidates.append(int(hits[0]) if len(hits) else None)
    valid=[v for v in candidates if v is not None]; fallback=int(round(np.median(valid))) if valid else int(round((lo+hi)/2)); onsets=np.array([v if v is not None else fallback for v in candidates])
    centered=x-np.median(x,axis=-1,keepdims=True); finite=np.isfinite(centered); safe=np.where(finite,centered,np.nan)
    channel_sd=np.nanstd(safe,axis=(0,2)); channel_med=np.nanmedian(safe,axis=(0,2)); channel_mad=1.4826*np.nanmedian(np.abs(safe-channel_med[None,:,None]),axis=(0,2))
    scale=np.maximum(channel_mad,1e-12); extreme=np.abs(safe-channel_med[None,:,None])>CONFIG["extreme_robust_z"]*scale[None,:,None]
    ptp=np.nanmax(safe,axis=-1)-np.nanmin(safe,axis=-1); variances=np.nanvar(safe,axis=(0,2)); logvar=np.log(np.maximum(variances,1e-20))
    freqs,psd=signal.welch(np.nan_to_num(centered),fs=CONFIG["native_sfreq"],nperseg=CONFIG["qc_welch_nperseg"],axis=-1)
    line_ratio=ratio(psd,freqs,(49,51),(45,55)); hf_ratio=ratio(psd,freqs,(30,40),(8,30))
    sos=signal.butter(4,CONFIG["qc_bandpass_hz"],btype="bandpass",fs=CONFIG["native_sfreq"],output="sos"); filt=signal.sosfiltfilt(sos,np.nan_to_num(centered),axis=-1)
    motor_idx=[CONFIG["liu29_names"].index(ch) for ch in CONFIG["motor13_names"]]; n=int(CONFIG["window_seconds"]*CONFIG["native_sfreq"]); motor=np.stack([trial[motor_idx,onset:onset+n] for trial,onset in zip(filt,onsets)])
    ranks=[]; conditions=[]
    for trial in motor:
        cov=OAS().fit(trial.T).covariance_; eig=np.maximum(np.linalg.eigvalsh(cov),1e-20); p=eig/eig.sum(); ranks.append(float(np.exp(-(p*np.log(p)).sum()))); conditions.append(float(eig.max()/eig.min()))
    bands=[(8,12),(13,20),(21,30)]; bf,bpsd=signal.welch(motor,fs=CONFIG["native_sfreq"],nperseg=CONFIG["qc_welch_nperseg"],axis=-1)
    logbp=np.stack([np.log(np.maximum(bpsd[...,((bf>=a)&(bf<b))].sum(-1),1e-20)) for a,b in bands],axis=-1)
    first=logbp[::2].mean(0).ravel(); second=logbp[1::2].mean(0).ravel(); reliability=float(np.corrcoef(first,second)[0,1]) if np.std(first)>0 and np.std(second)>0 else np.nan
    return {"subject_id":sid,"qc_scope":"full_recording_transductive_label_blind","n_trials":int(len(x)),"marker_valid_count":int(len(valid)),"marker_fallback_count":int(len(x)-len(valid)),"marker_valid_fraction":float(len(valid)/len(x)),"finite_fraction":float(finite.mean()),"channel_sd_median":float(np.nanmedian(channel_sd)),"robust_sd_median":float(np.nanmedian(channel_mad)),"robust_sd_min":float(np.nanmin(channel_mad)),"robust_sd_max":float(np.nanmax(channel_mad)),"peak_to_peak_median":float(np.nanmedian(ptp)),"peak_to_peak_p95":float(np.nanpercentile(ptp,95)),"clipped_fraction":float(np.nanmean(np.abs(safe)>=CONFIG["adc_centered_rail"])),"extreme_fraction":float(np.nanmean(extreme)),"flat_channel_count":int(np.sum(channel_mad<CONFIG["flat_scale_min"])),"line_50hz_ratio":line_ratio,"hf_30_40_to_8_30_ratio":hf_ratio,"channel_variance_log_iqr":float(np.nanpercentile(logvar,75)-np.nanpercentile(logvar,25)),"motor13_cov_effective_rank_mean":float(np.mean(ranks)),"motor13_cov_condition_median":float(np.median(conditions)),"split_half_log_bandpower_reliability":reliability}
QC_SIGNATURE,QC_SIGNATURE_PAYLOAD=qc_signature(); QC_CACHE_DIR=Path(CONFIG["artifact_dir"])/"qc_cache"/f"qc_{QC_SIGNATURE}"; QC_CACHE_DIR.mkdir(parents=True,exist_ok=True); qc_cache_path=QC_CACHE_DIR/"qc_features.csv"
if qc_cache_path.exists(): QC=pd.read_csv(qc_cache_path); print(f"Loaded QC cache: {qc_cache_path}")
else:
    QC=pd.DataFrame([compute_subject_qc(sid) for sid in expected]); QC.to_csv(qc_cache_path,index=False); (QC_CACHE_DIR/"signature.json").write_text(json.dumps(QC_SIGNATURE_PAYLOAD,indent=2,default=str),encoding="utf-8"); print(f"Created QC cache: {qc_cache_path}")
if sorted(QC["subject_id"].tolist())!=expected or QC["subject_id"].duplicated().any(): raise AssertionError("QC cache subject mismatch")
qc_features_path=ARTIFACT_DIR/"qc_features.csv"; QC.to_csv(qc_features_path,index=False)

# 4. Analysis Model
## 4.1 Fixed Outcome-Blind QC Policies
## 4.2 Nested Patient Reliability Selector
Ridge preprocessing and alpha tuning use development patients only; held-out scores are persisted before outcome evaluation.

In [ ]:
ANALYSIS=SUBJECT_BA.merge(clinical,on="subject_id",validate="one_to_one").merge(QC,on="subject_id",validate="one_to_one")
policy_expr={
 "marker_valid":(ANALYSIS.marker_valid_fraction>=CONFIG["marker_valid_fraction_min"])&(ANALYSIS.marker_fallback_count<=CONFIG["marker_fallback_max"]),
 "all_finite":ANALYSIS.finite_fraction==1.0,
 "robust_scale_plausible":ANALYSIS.robust_sd_median.between(*CONFIG["robust_scale_range"]),
 "no_flat_channels":ANALYSIS.flat_channel_count==0,
 "peak_to_peak_plausible":ANALYSIS.peak_to_peak_median<=CONFIG["peak_to_peak_max"],
 "extreme_fraction_plausible":ANALYSIS.extreme_fraction<=CONFIG["extreme_fraction_max"],
 "clipped_fraction_plausible":ANALYSIS.clipped_fraction<=CONFIG["clipped_fraction_max"],
 "covariance_numerically_valid":np.isfinite(ANALYSIS.motor13_cov_condition_median)&(ANALYSIS.motor13_cov_condition_median<=CONFIG["cov_condition_max"])&(ANALYSIS.motor13_cov_effective_rank_mean>=CONFIG["cov_effective_rank_min"])
}
for name,mask in policy_expr.items(): ANALYSIS[f"qc_pass_{name}"]=np.asarray(mask,bool)
policy_cols=[f"qc_pass_{k}" for k in policy_expr]; ANALYSIS["qc_pass_combined_fixed"]=ANALYSIS[policy_cols].all(axis=1); ANALYSIS["hard_hardware_failure"]=(~ANALYSIS.qc_pass_all_finite)|(~ANALYSIS.qc_pass_no_flat_channels)
coverage=float(ANALYSIS.qc_pass_combined_fixed.mean()); rejected=ANALYSIS.loc[~ANALYSIS.qc_pass_combined_fixed]
if coverage<CONFIG["minimum_policy_coverage"] and not (len(rejected)>0 and rejected.hard_hardware_failure.all()): raise AssertionError(f"Fixed QC retained {coverage:.1%}, below 70%; do not tune thresholds on BA")
POLICY_COUNTS={name:{"pass":int(mask.sum()),"fail":int((~mask).sum())} for name,mask in policy_expr.items()}; POLICY_COUNTS["combined_fixed"]={"pass":int(ANALYSIS.qc_pass_combined_fixed.sum()),"fail":int((~ANALYSIS.qc_pass_combined_fixed).sum())}
print(json.dumps(POLICY_COUNTS,indent=2))

In [ ]:
features=CONFIG["selector_features"]
if len(features)>8 or not np.isfinite(ANALYSIS[features].to_numpy(float)).all(): raise AssertionError("Selector requires at most 8 finite prespecified QC features")
outer=KFold(n_splits=CONFIG["selector_outer_folds"],shuffle=True,random_state=BASE_SEED); selector_rows=[]
for outer_fold,(dev,test) in enumerate(outer.split(ANALYSIS)):
    seed_everything(BASE_SEED+outer_fold); inner=KFold(n_splits=CONFIG["selector_inner_folds"],shuffle=True,random_state=BASE_SEED+100+outer_fold)
    pipe=Pipeline([("scale",StandardScaler()),("ridge",Ridge())]); search=GridSearchCV(pipe,{"ridge__alpha":CONFIG["ridge_alphas"]},cv=inner,scoring="neg_mean_absolute_error",refit=True)
    search.fit(ANALYSIS.iloc[dev][features],ANALYSIS.iloc[dev]["balanced_accuracy"]); score=search.predict(ANALYSIS.iloc[test][features])
    for idx,predicted in zip(test,score): selector_rows.append({"subject_id":ANALYSIS.iloc[idx].subject_id,"outer_fold":outer_fold,"predicted_reliability":float(predicted),"selected_alpha":float(search.best_params_["ridge__alpha"]),"n_development_patients":int(len(dev)),"score_status":"written_before_outcome_evaluation"})
PRE_EVAL_SCORES=pd.DataFrame(selector_rows).sort_values("subject_id"); pre_eval_path=ARTIFACT_DIR/"selector_scores_pre_evaluation.csv"; PRE_EVAL_SCORES.to_csv(pre_eval_path,index=False)
pre_eval_sha256=hashlib.sha256(pre_eval_path.read_bytes()).hexdigest(); print(f"Held-out selector scores written before evaluation: {pre_eval_path} sha256={pre_eval_sha256}")
SELECTOR=PRE_EVAL_SCORES.merge(SUBJECT_BA[["subject_id","balanced_accuracy"]],on="subject_id",validate="one_to_one"); SELECTOR["residual"]=SELECTOR.balanced_accuracy-SELECTOR.predicted_reliability
selector_predictions_path=ARTIFACT_DIR/"selector_predictions.csv"; SELECTOR.to_csv(selector_predictions_path,index=False)
rho,rho_p=spearmanr(SELECTOR.predicted_reliability,SELECTOR.balanced_accuracy); slope,intercept=np.polyfit(SELECTOR.predicted_reliability,SELECTOR.balanced_accuracy,1)
SELECTOR_METRICS={"r2":float(r2_score(SELECTOR.balanced_accuracy,SELECTOR.predicted_reliability)),"mae":float(mean_absolute_error(SELECTOR.balanced_accuracy,SELECTOR.predicted_reliability)),"spearman_rho":float(rho),"spearman_p":float(rho_p),"calibration_slope":float(slope),"calibration_intercept":float(intercept),"pre_evaluation_scores_sha256":pre_eval_sha256}

# 5. Analysis
## 5.1 Fixed Cohorts and Clinical Strata
## 5.2 Cross-Fitted Coverage-Risk Analysis
## 5.3 `oracle_invalid`: Circular Outcome-Ranked Demonstration
This analysis is invalid, circular, and not deployable. It is computed only to demonstrate manufactured high accuracy.
## 5.4 Posthoc Uncalibrated Trial-Confidence Abstention

In [ ]:
lv=set(CONFIG["lv14_subjects"]); cohort_defs={"Full50":ANALYSIS.index==ANALYSIS.index,"Lv14_fixed":ANALYSIS.subject_id.isin(lv),"non_Lv36":~ANALYSIS.subject_id.isin(lv),"fixed_QC_combined":ANALYSIS.qc_pass_combined_fixed}
cohort_rows=[]
for i,(name,mask) in enumerate(cohort_defs.items()):
    frame=ANALYSIS.loc[mask]; ci=bootstrap_ci(frame.balanced_accuracy,i)
    cohort_rows.append({"cohort":name,"analysis_role":CONFIG["analysis_role"],"n_subjects":len(frame),"coverage_fraction":float(len(frame)/len(ANALYSIS)),"mean_subject_ba":float(frame.balanced_accuracy.mean()),"ci95_low":ci[0],"ci95_high":ci[1],"selection_status":"fixed_outcome_blind"})
COHORT_RESULTS=pd.DataFrame(cohort_rows); cohort_results_path=ARTIFACT_DIR/"cohort_results.csv"; COHORT_RESULTS.to_csv(cohort_results_path,index=False)
strata_spec=[("paralysis_side","ParalysisSide",sorted(ANALYSIS.ParalysisSide.unique())),("nihss","nihss_category",CONFIG["nihss_labels"]),("age","age_category",CONFIG["age_labels"]),("duration","duration_category",CONFIG["duration_labels"]),("mRS","mRS_category",CONFIG["mrs_labels"]),("MBI","mbi_category",CONFIG["mbi_labels"])]
strata_rows=[]; omnibus=[]
for family_i,(family,col,categories) in enumerate(strata_spec):
    groups=[]
    for category_i,category in enumerate(categories):
        vals=ANALYSIS.loc[ANALYSIS[col].astype(str)==str(category),"balanced_accuracy"].to_numpy(); ci=bootstrap_ci(vals,100+family_i*20+category_i)
        strata_rows.append({"clinical_family":family,"category":str(category),"n_subjects":len(vals),"coverage_fraction":float(len(vals)/50),"mean_subject_ba":float(vals.mean()) if len(vals) else np.nan,"ci95_low":ci[0],"ci95_high":ci[1]})
        if len(vals): groups.append(vals)
    stat,p=kruskal(*groups) if len(groups)>=2 else (np.nan,np.nan); omnibus.append({"clinical_family":family,"kruskal_statistic":float(stat),"p_value":float(p)})
omnibus_df=pd.DataFrame(omnibus); omnibus_df["holm_p"]=holm_adjust(omnibus_df.p_value); CLINICAL_STRATA=pd.DataFrame(strata_rows).merge(omnibus_df,on="clinical_family",validate="many_to_one"); CLINICAL_STRATA["holm_family"]="six_prespecified_clinical_omnibus_tests"
clinical_strata_path=ARTIFACT_DIR/"clinical_strata.csv"; CLINICAL_STRATA.to_csv(clinical_strata_path,index=False)

In [ ]:
scored=ANALYSIS.merge(PRE_EVAL_SCORES[["subject_id","predicted_reliability"]],on="subject_id",validate="one_to_one")
def partition_row(curve,coverage,partition,frame,rank_variable,status,seed_offset):
    ci=bootstrap_ci(frame.balanced_accuracy,seed_offset)
    return {"curve":curve,"target_coverage":coverage,"partition":partition,"n_subjects":len(frame),"achieved_coverage":float(len(frame)/50),"mean_subject_ba":float(frame.balanced_accuracy.mean()) if len(frame) else np.nan,"ci95_low":ci[0],"ci95_high":ci[1],"rank_variable":rank_variable,"validity_status":status,"clinical_composition_json":json.dumps(clinical_composition(frame),sort_keys=True)}
coverage_rows=[]
ranked=scored.sort_values(["predicted_reliability","subject_id"],ascending=[False,True])
for i,cov in enumerate(CONFIG["coverage_levels"]):
    n=int(np.ceil(50*cov)); retained=ranked.iloc[:n]; rejected=ranked.iloc[n:]
    coverage_rows += [partition_row("cross_fitted_selector",cov,"retained",retained,"cross_fitted_predicted_reliability","nested_descriptive",200+i*2),partition_row("cross_fitted_selector",cov,"rejected",rejected,"cross_fitted_predicted_reliability","nested_descriptive",201+i*2)]
COVERAGE_RISK=pd.DataFrame(coverage_rows); coverage_risk_path=ARTIFACT_DIR/"coverage_risk.csv"; COVERAGE_RISK.to_csv(coverage_risk_path,index=False)
oracle_ranked=ANALYSIS.sort_values(["balanced_accuracy","subject_id"],ascending=[False,True]); oracle_rows=[]
for i,cov in enumerate(CONFIG["oracle_coverage_levels"]):
    n=int(np.ceil(50*cov)); retained=oracle_ranked.iloc[:n]; rejected=oracle_ranked.iloc[n:]
    oracle_rows += [partition_row("oracle_invalid",cov,"retained",retained,"observed_held_out_ba","INVALID_CIRCULAR_NOT_DEPLOYABLE",300+i*2),partition_row("oracle_invalid",cov,"rejected",rejected,"observed_held_out_ba","INVALID_CIRCULAR_NOT_DEPLOYABLE",301+i*2)]
means=np.array([oracle_ranked.iloc[:k].balanced_accuracy.mean() for k in range(1,51)]); hits=np.flatnonzero(means>=CONFIG["oracle_demonstration_target_ba"]); manufactured_n=int(hits[-1]+1) if len(hits) else 0
if manufactured_n:
    manufactured=oracle_ranked.iloc[:manufactured_n]; oracle_rows.append(partition_row("oracle_invalid_75pct_demonstration",manufactured_n/50,"retained",manufactured,"observed_held_out_ba","INVALID_CIRCULAR_MANUFACTURED_NOT_DEPLOYABLE",399))
ORACLE_INVALID=pd.DataFrame(oracle_rows); oracle_invalid_path=ARTIFACT_DIR/"oracle_invalid_curve.csv"; ORACLE_INVALID.to_csv(oracle_invalid_path,index=False)
print(f"INVALID ORACLE ONLY: largest top-ranked prefix at or above 75% BA retains {manufactured_n}/50 patients")

In [ ]:
ranked_trials=TRIAL_OOF.sort_values(["confidence","subject_id","trial_index"],ascending=[False,True,True]); abstention_rows=[]
for target in CONFIG["trial_abstention_levels"]:
    n=int(np.ceil(len(ranked_trials)*target)); kept=ranked_trials.iloc[:n]; present=sorted(kept.true_label.unique()); ba=float(balanced_accuracy_score(kept.true_label,kept.prediction)) if present==[0,1] else np.nan
    class_cov={c:float((kept.true_label==c).sum()/(TRIAL_OOF.true_label==c).sum()) for c in [0,1]}; sens={c:(float((kept.loc[kept.true_label==c,"prediction"]==c).mean()) if (kept.true_label==c).any() else np.nan) for c in [0,1]}
    subject_bas=[]; undefined=0
    for sid,group in kept.groupby("subject_id"):
        if set(group.true_label)=={0,1}: subject_bas.append(balanced_accuracy_score(group.true_label,group.prediction))
        else: undefined+=1
    abstention_rows.append({"target_coverage":target,"n_trials_retained":n,"achieved_coverage":float(n/len(ranked_trials)),"confidence_threshold":float(kept.confidence.min()),"selective_pooled_ba":ba,"mean_defined_subject_ba":float(np.mean(subject_bas)) if subject_bas else np.nan,"n_subjects_with_defined_ba":len(subject_bas),"n_subjects_undefined_missing_class":undefined,"class0_coverage":class_cov[0],"class1_coverage":class_cov[1],"class0_sensitivity":sens[0],"class1_sensitivity":sens[1],"analysis_status":"POSTHOC_DESCRIPTIVE_UNCALIBRATED_CONFIDENCE"})
TRIAL_ABSTENTION=pd.DataFrame(abstention_rows); trial_abstention_path=ARTIFACT_DIR/"trial_abstention.csv"; TRIAL_ABSTENTION.to_csv(trial_abstention_path,index=False)

# 6. Results
## 6.1 Aggregate Metrics
## 6.2 Performance Visualizations

In [ ]:
plot_paths=[]
fig,ax=plt.subplots(figsize=(7,4)); ax.bar(COHORT_RESULTS.cohort,100*COHORT_RESULTS.mean_subject_ba,color=["#355070","#6d597a","#b56576","#2a9d8f"]); ax.axhline(50,color="black",ls="--"); ax.set_ylabel("Mean subject BA (%)"); ax.tick_params(axis="x",rotation=20); fig.tight_layout(); p=ARTIFACT_DIR/"cohort_balanced_accuracy.png"; fig.savefig(p,dpi=160); plt.close(fig); plot_paths.append(p)
fig,ax=plt.subplots(figsize=(6,5)); ax.scatter(SELECTOR.predicted_reliability,SELECTOR.balanced_accuracy,color="#355070"); lo=min(SELECTOR.predicted_reliability.min(),SELECTOR.balanced_accuracy.min()); hi=max(SELECTOR.predicted_reliability.max(),SELECTOR.balanced_accuracy.max()); ax.plot([lo,hi],[lo,hi],ls="--",color="black"); ax.set(xlabel="Cross-fitted predicted reliability",ylabel="Observed locked OOF BA",title="Nested patient reliability selector"); fig.tight_layout(); p=ARTIFACT_DIR/"selector_calibration.png"; fig.savefig(p,dpi=160); plt.close(fig); plot_paths.append(p)
fig,ax=plt.subplots(figsize=(7,4)); valid=COVERAGE_RISK.query("partition == 'retained'"); invalid=ORACLE_INVALID.query("partition == 'retained' and curve == 'oracle_invalid'"); ax.plot(100*valid.achieved_coverage,100*valid.mean_subject_ba,marker="o",label="Cross-fitted selector",color="#2a9d8f"); ax.plot(100*invalid.achieved_coverage,100*invalid.mean_subject_ba,marker="x",label="ORACLE INVALID / CIRCULAR",color="#c1121f"); ax.axhline(75,color="#c1121f",ls=":"); ax.set(xlabel="Patient coverage (%)",ylabel="Retained mean subject BA (%)"); ax.legend(); fig.tight_layout(); p=ARTIFACT_DIR/"coverage_risk_with_invalid_oracle.png"; fig.savefig(p,dpi=160); plt.close(fig); plot_paths.append(p)
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(100*TRIAL_ABSTENTION.achieved_coverage,100*TRIAL_ABSTENTION.selective_pooled_ba,marker="o",color="#6d597a"); ax.set(xlabel="Trial coverage (%)",ylabel="Selective pooled BA (%)",title="Posthoc uncalibrated confidence abstention"); fig.tight_layout(); p=ARTIFACT_DIR/"trial_confidence_abstention.png"; fig.savefig(p,dpi=160); plt.close(fig); plot_paths.append(p)
qc_plot_cols=["robust_sd_median","peak_to_peak_median","line_50hz_ratio","split_half_log_bandpower_reliability"]; fig,axes=plt.subplots(2,2,figsize=(8,6));
for ax,col in zip(axes.ravel(),qc_plot_cols): ax.hist(ANALYSIS[col],bins=10,color="#457b9d"); ax.set_title(col)
fig.suptitle("Label-blind full-recording QC (transductive)"); fig.tight_layout(); p=ARTIFACT_DIR/"qc_feature_distributions.png"; fig.savefig(p,dpi=160); plt.close(fig); plot_paths.append(p)

## 6.3 Experiment Summary

In [ ]:
full=COHORT_RESULTS.set_index("cohort").loc["Full50"]; lv14=COHORT_RESULTS.set_index("cohort").loc["Lv14_fixed"]; nonlv=COHORT_RESULTS.set_index("cohort").loc["non_Lv36"]
GLOBAL_METRICS={"analysis_role":CONFIG["analysis_role"],"source_artifact":str(SOURCE_ARTIFACT),"validation":{"subject_ids_exact":True,"n_subjects":50,"trials_per_subject":40,"n_exactly_once_oof_predictions":2000},"cohorts":{"Full50":full.to_dict(),"Lv14_fixed":lv14.to_dict(),"non_Lv36":nonlv.to_dict()},"qc":{"scope":"full_recording_transductive_label_blind_not_fold_local","signature":QC_SIGNATURE,"cache_dir":str(QC_CACHE_DIR),"policy_counts":POLICY_COUNTS,"combined_coverage":coverage},"clinical_holm_family":omnibus_df.to_dict(orient="records"),"selector":SELECTOR_METRICS,"oracle_invalid":{"status":"INVALID_CIRCULAR_NOT_DEPLOYABLE","target_ba":CONFIG["oracle_demonstration_target_ba"],"largest_prefix_at_or_above_target_n":manufactured_n},"trial_abstention_status":"POSTHOC_DESCRIPTIVE_UNCALIBRATED_CONFIDENCE"}
print(json.dumps(json_safe(GLOBAL_METRICS),indent=2))

## 6.4 Validity Diagnostics
The valid selector is cross-fitted across patients. Full-recording QC remains transductive; the oracle is invalid/circular; confidence abstention is posthoc and uncalibrated.
## 6.5 Save Artifacts

In [ ]:
global_metrics_path=ARTIFACT_DIR/"global_metrics.json"; global_metrics_path.write_text(json.dumps(json_safe(GLOBAL_METRICS),indent=2,allow_nan=False),encoding="utf-8")
run_metadata_path=ARTIFACT_DIR/"run_metadata.json"; artifact_index={p.name:str(p) for p in ARTIFACT_DIR.iterdir()}; artifact_index[run_metadata_path.name]=str(run_metadata_path)
run_metadata={"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"experiment_name":CONFIG["experiment_name"],"config_note":CONFIG["config_note"],"analysis_role":CONFIG["analysis_role"],"source_artifact":str(SOURCE_ARTIFACT),"subjects":expected,"channel_names":CONFIG["liu29_names"],"seed":BASE_SEED,"global_metrics":json_safe(GLOBAL_METRICS),"artifacts":artifact_index,"qc_cache":{"signature":QC_SIGNATURE,"path":str(QC_CACHE_DIR)},"validity_warnings":["QC is full-recording transductive, not strict fold-local.","oracle_invalid is circular and not deployable.","Trial confidence abstention is posthoc and uncalibrated."]}
run_metadata_path.write_text(json.dumps(run_metadata,indent=2,allow_nan=False),encoding="utf-8")
required=[cohort_results_path,qc_features_path,selector_predictions_path,coverage_risk_path,oracle_invalid_path,clinical_strata_path,trial_abstention_path,global_metrics_path,*plot_paths]
missing=[str(p) for p in required if not Path(p).exists()];
if missing: raise AssertionError(f"Missing required artifacts: {missing}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass